# Living Lab radar + pose: a student integration guide

**09_SY · 19_MM · SyncWB training export v3**

This notebook accompanies the immutable HDF5 export. Read it in order, then run
the cells to explore your own frames. The plots and tables saved here were executed
against the accompanying files. All data access is read-only. No SyncWB database,
raw ADC files, original recordings, or ML checkout is needed for the main examples.

You will learn to:
1. Verify the package and inspect every array and its meaning.
2. Load variable-length clouds and independently ordered 2D/3D people.
3. Plot both subjects, inspect timing, counts and confidence, and understand calibration.
4. Construct sequence/frame catalogues and prepare a target for the VIP HPE contracts.
5. Identify the work still needed in the ML adapter, experiment assembly and evaluation.

**Read these facts before training:** this release uses initial timestamp alignment,
not a manually refined synchronization model. Kinect poses are estimated supervision,
not motion-capture ground truth. 2D and 3D person rows have no shared identity.
The exported XYZ values are in an approximate floor/world frame; the existing ML
radar contracts use `radar_link`. The notebook explicitly inverts that transform.

Navigation: [files](#1.-Locate-and-verify-the-package) · [fields](#2.-The-complete-array-contract)
· [load](#3.-Read-one-radar-frame-and-its-pose-frame) · [plots](#4.-Point-cloud-and-3D-pose-plots)
· [timing](#6.-Timing,-gaps-and-correspondence) · [calibration](#8.-Calibration-and-coordinate-conversion)
· [integration](#10.-Build-the-ML-sequence-and-frame-catalogues).

## 1. Locate and verify the package

Keep this notebook beside `manifest.json`. Set `DATA_ROOT` explicitly if your current
working directory is elsewhere. Paths to payloads are resolved from the manifest;
never infer subjects from hashed filenames.

The six original files are `manifest.json`, a brief `README.txt`, and one radar/pose
HDF5 pair per run pair. This notebook is accompanying documentation and is **not** a
hashed training payload. Keep the original manifest and HDF5 files unchanged.
The package contains offline clouds derived from raw radar recordings, **not ADC
waveforms or online clouds**, plus the imported Kinect/RTMPose outputs. RGB video
is referenced, not included. No training windows, train/test split or padding is stored.

In [ ]:
from pathlib import Path
import hashlib
import json
from contextlib import contextmanager
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

DATA_ROOT = Path.cwd()  # Change only this path if the notebook is opened from elsewhere.
assert (DATA_ROOT / 'manifest.json').is_file(), 'Set DATA_ROOT to the directory containing manifest.json.'
DATA_ROOT = DATA_ROOT.resolve()
manifest = json.loads((DATA_ROOT / 'manifest.json').read_text(encoding='utf-8'))
assert manifest['schema'] == 'syncwb.training_export.v3', manifest['schema']
segments = {s['subject_id']: s for s in manifest['segments']}
assert len(segments) == len(manifest['segments']), 'This teaching preset expects one run pair per subject.'

def package_path(ref):
    path = (DATA_ROOT / ref).resolve()
    if not path.is_relative_to(DATA_ROOT) or not path.is_file():
        raise ValueError(f'Missing or escaping package path: {ref}')
    return path

@contextmanager
def open_pair(subject):
    s = segments[subject]
    with h5py.File(package_path(s['files']['radar']['path']), 'r') as r, \
         h5py.File(package_path(s['files']['pose']['path']), 'r') as p:
        yield r, p

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

plt.rcParams.update({'figure.dpi': 110, 'font.size': 10, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.titleweight': 'bold'})
print('Format:', manifest['schema'], '| subjects:', ', '.join(manifest['subjects']))
print('Dataset fingerprint:', manifest['dataset_fingerprint'])
display(pd.DataFrame([{'subject': s['subject_id'], 'radar_frames': s['radar_frames'],
                      'Kinect_frames': s['pose_frames'], 'radar_file': s['files']['radar']['path'],
                      'pose_file': s['files']['pose']['path']} for s in manifest['segments']]))

In [ ]:
# Verify all four payloads by default. This reads the files sequentially, not into RAM.
VERIFY_PAYLOAD_HASHES = True
fingerprint_content = {k: v for k, v in manifest.items() if k not in ('dataset_fingerprint', 'created_at')}
canonical = json.dumps(fingerprint_content, sort_keys=True, separators=(',', ':'), ensure_ascii=False)
assert hashlib.sha256(canonical.encode()).hexdigest() == manifest['dataset_fingerprint']
checks = []
for s in manifest['segments']:
    for modality, entry in s['files'].items():
        path = package_path(entry['path'])
        ok = sha256_file(path) == entry['sha256'] if VERIFY_PAYLOAD_HASHES else None
        if VERIFY_PAYLOAD_HASHES:
            assert ok, f'Checksum mismatch: {path.name}'
        checks.append({'subject': s['subject_id'], 'modality': modality,
                       'MiB': round(path.stat().st_size / 2**20, 2), 'SHA256_verified': ok})
display(pd.DataFrame(checks))
print('Manifest fingerprint verified. Hashes detect changes; they are not digital signatures.')

## 2. The complete array contract

There are three different indices. **HDF5 row** selects a row in the exported file.
`frames/sample_index` is the original zero-based canonical acquisition index.
`frames/frame_number` is the source recording's frame number (or −1 if unknown).
They are not interchangeable: the first exported 19_MM radar row has native sample
index 4750. Kinect source frame numbers in these runs are one-based; MMPose JSON
`frame_id` is zero-based. Read the fields rather than guessing an offset.

Radar and Kinect have different frame counts/cadences. A radar row maps through
`correspondence/pose_row` to a **Kinect HDF5 row**, never by the same row number.
Multiple radar rows can use the same pose. `-1` means unmatched and must be checked
**before indexing**; NumPy would otherwise quietly select the last pose row.

`points`, `poses`, and `poses2d` are ragged groups. For frame row `i`, each group's
`offsets[i]:offsets[i+1]` selects its points or people. 3D and 2D offsets are independent.
Empty slices can mean available-but-empty or unavailable; inspect the frame flags.
Counts of −1 mean unknown/unprocessed, not zero. Missing PTS/bbox confidence is NaN.
Finiteness masks test coordinates only, not confidence or anatomical correctness.

In [ ]:
FIELD_HELP = {'frames/sample_index': 'Original zero-based canonical acquisition index; preserve in source_locator.', 'frames/frame_number': 'Original source frame number; -1 if unknown. Not an HDF5 row.', 'frames/timeline_time_s': 'Selected original modality timeline minus its run timeline_origin (seconds).', 'frames/aligned_time_s': 'Time in common radar coordinates minus shared aligned_origin (seconds).', 'frames/sequence_block': 'Shared continuity partition. Windows and matches cannot cross it.', 'frames/payload_available': 'Radar: processed payload exists, even if empty. Pose: 3D payload exists, even if empty.', 'frames/point_count': 'Number of points including every association code; -1 for unprocessed.', 'frames/point_status': 'ASCII available or unprocessed. Decode bytes if needed.', 'points/offsets': 'N_radar + 1 int64 ragged boundaries into points/values.', 'points/values': 'Float32 [total_points,6]: world x,y,z (m), radial_velocity (m/s), SNR (linear power ratio), association_id.', 'correspondence/pose_row': 'Matched paired-pose HDF5 row; -1 for no supported match.', 'correspondence/pose_sample_index': 'Original Kinect canonical sample index of the match; -1 if unmatched.', 'correspondence/matched': 'A supported temporal match exists; not a label-quality guarantee.', 'correspondence/nearest_residual_ms': 'Nearest candidate predicted Kinect time minus radar time, ms, even if rejected.', 'correspondence/pose_payload_available': 'Matched frame contains at least one stored 3D pose. Does not test confidence or 2D.', 'frames/num_people': 'Imported annotated presence count, not a detector count; -1 if unknown.', 'frames/num_2d': 'Imported 2D detection summary count; -1 if unknown.', 'frames/num_3d': 'Imported 3D detection summary count; -1 if unknown.', 'frames/pose_person_count': 'Actual stored 3D people in the frame, from ragged length.', 'frames/pose2d_person_count': 'Actual stored 2D detections, independently ordered.', 'frames/pose2d_payload_available': '2D payload exists, including an empty array.', 'frames/conf2d_payload_available': 'Bounding-box-score payload exists, including an empty array.', 'frames/rgb_pts_s': 'Original RGB presentation timestamp (seconds); NaN if absent. Not radar aligned time.', 'frames/activity_json': 'UTF-8 JSON string per Kinect frame; dictionary of source tiers or null.', 'frames/activity_payload_available': 'Activity source payload exists; empty tiers are not missing data.', 'poses/offsets': 'N_Kinect + 1 boundaries into all 3D person arrays.', 'poses/xyz': 'Float32 [total_3d_people,32,3], calibrated common world metres.', 'poses/confidence': 'Float32 [total_3d_people,32], native Kinect confidence levels, not probabilities.', 'poses/finite_xyz': 'Bool [total_3d_people,32], all three coordinates finite.', 'poses2d/offsets': 'N_Kinect + 1 boundaries into all 2D person arrays; independent of poses/offsets.', 'poses2d/xy': 'Float64 [total_2d_people,26,2], original RTMPose pixel x/y.', 'poses2d/confidence': 'Float64 [total_2d_people,26], detector keypoint scores; not calibrated probabilities.', 'poses2d/bbox_confidence': 'Float64 [total_2d_people], detector bounding-box score; NaN if unavailable. No box coordinates.', 'poses2d/finite_xy': 'Bool [total_2d_people,26], both pixel coordinates finite.'}

inventory = []
subject = manifest['subjects'][0]
with open_pair(subject) as (r, p):
    for modality, h in [('radar', r), ('pose', p)]:
        def describe(name, obj):
            if isinstance(obj, h5py.Dataset):
                assert name in FIELD_HELP, f'Undocumented dataset: {name}'
                inventory.append({'file': modality, 'dataset': name, 'shape': str(obj.shape),
                                  'dtype': str(obj.dtype), 'meaning': FIELD_HELP[name]})
        h.visititems(describe)
with pd.option_context('display.max_rows', 80, 'display.max_colwidth', 160):
    display(pd.DataFrame(inventory))

**Attributes:** each HDF5 file has `schema`, `segment_id`, `subject_id`, `radar_run_id`,
`pose_run_id`, `point_cloud_version_id`, `mapping_version_id`, `processing_recipe_id`,
`processing_compatibility_id`, `mapping_profile_id`, `modality`, `timeline_model_id`,
`nominal_sample_period_s`, `timeline_origin`, `aligned_origin`, `time_coordinate_kind`,
and `geometry_json`. They identify data/provenance, define timing, and repeat geometry.

`points` has `columns_json` and `units_json`; `poses` has `joint_names_json`.
`poses2d` also has `schema`, `coordinate_frame`, `image_size_status` and
`person_correspondence`. `correspondence` has `pose_file`, `tolerance_ms` and
`residual_definition`. Parse attributes ending in `_json` with `json.loads`.
The original export correctly says 2D image size was not in canonical artifacts;
section 5 adds separately verified source-video evidence without editing that metadata.

In [ ]:
with open_pair(manifest['subjects'][0]) as (r, p):
    joint_names3 = json.loads(p['poses'].attrs['joint_names_json'])
    joint_names2 = json.loads(p['poses2d'].attrs['joint_names_json'])
    point_columns = json.loads(r['points'].attrs['columns_json'])
    point_units = json.loads(r['points'].attrs['units_json'])
    display(pd.DataFrame({'point_column': point_columns, 'unit': point_units}))
    display(pd.DataFrame({'joint_index': range(32), 'Kinect_3D': joint_names3,
                          'RTMPose_2D': joint_names2 + ['—'] * 6}))
    print('Side-by-side joint lists show index order only; they are NOT a 2D-to-3D mapping.')

## 3. Read one radar frame and its pose frame

This loader returns all people and all points, with no confidence threshold,
crop, resampling, association filtering or accumulation. Only the requested
ragged slices are loaded. The helper closes each HDF5 file after use.

The `people` annotation can differ from both detection counts. No persistent Kinect
body IDs or 2D↔3D identities are exported. Radar association codes do not identify
a Kinect person. These generators label 250=`ghost_likely`, 251=`ghost_behind`,
252=`reserved`, 253=`too_weak`, 254=`outside_boundary`, 255=`not_associated` in
their embedded `generator_metadata.association_labels`. Keep those non-track codes
distinct from real tracker IDs; do not feed the column as a continuous feature or
cast every nonnegative code into a valid track ID. Track IDs are run-local and do
not establish persistent human identity across recordings.

In [ ]:
def ragged(h, group, field, row):
    lo, hi = map(int, h[f'{group}/offsets'][row:row + 2])
    return h[f'{group}/{field}'][lo:hi]

def read_pair(subject, radar_row):
    s = segments[subject]
    with open_pair(subject) as (r, p):
        row = int(radar_row)
        if not 0 <= row < len(r['frames/sample_index']):
            raise IndexError(row)
        sample = dict(subject=subject, segment_id=s['segment_id'], radar_row=row,
                      radar_sample_index=int(r['frames/sample_index'][row]),
                      radar_frame_number=int(r['frames/frame_number'][row]),
                      aligned_time_s=float(r['frames/aligned_time_s'][row]),
                      block=int(r['frames/sequence_block'][row]),
                      radar_available=bool(r['frames/payload_available'][row]),
                      points=ragged(r, 'points', 'values', row),
                      residual_ms=float(r['correspondence/nearest_residual_ms'][row]),
                      pose=None)
        pr = int(r['correspondence/pose_row'][row])
        if pr >= 0:
            assert bool(r['correspondence/matched'][row])
            assert sample['block'] == int(p['frames/sequence_block'][pr])
            sample['pose'] = dict(row=pr, sample_index=int(p['frames/sample_index'][pr]),
                frame_number=int(p['frames/frame_number'][pr]),
                aligned_time_s=float(p['frames/aligned_time_s'][pr]),
                rgb_pts_s=float(p['frames/rgb_pts_s'][pr]),
                people=int(p['frames/num_people'][pr]),
                available3d=bool(p['frames/payload_available'][pr]),
                available2d=bool(p['frames/pose2d_payload_available'][pr]),
                xyz=ragged(p, 'poses', 'xyz', pr),
                confidence3d=ragged(p, 'poses', 'confidence', pr),
                finite3d=ragged(p, 'poses', 'finite_xyz', pr),
                xy=ragged(p, 'poses2d', 'xy', pr),
                confidence2d=ragged(p, 'poses2d', 'confidence', pr),
                bbox_confidence=ragged(p, 'poses2d', 'bbox_confidence', pr),
                finite2d=ragged(p, 'poses2d', 'finite_xy', pr),
                activity_available=bool(p['frames/activity_payload_available'][pr]),
                activity=json.loads(p['frames/activity_json'][pr]))
        return sample

def choose_demonstration_row(subject):
    # Choose an inspectable single-person example; this is NOT a training membership policy.
    with open_pair(subject) as (r, p):
        pr = r['correspondence/pose_row'][:]
        safe = pr.clip(0)
        keep = ((pr >= 0) & r['frames/payload_available'][:] & (r['frames/point_count'][:] > 30)
                & (p['frames/num_people'][:][safe] == 1)
                & (p['frames/pose_person_count'][:][safe] == 1)
                & (p['frames/pose2d_person_count'][:][safe] == 1))
        candidates = np.flatnonzero(keep)
    assert len(candidates), f'No demonstration candidates for {subject}'
    middle = len(candidates) // 2
    for position in sorted(range(len(candidates)), key=lambda i: abs(i-middle))[:1000]:
        sample = read_pair(subject, int(candidates[position]))
        pose = sample['pose']
        if pose['finite3d'][0, 0] and pose['confidence3d'][0, 0] >= 2:
            return int(candidates[position])
    raise ValueError('Choose a demonstration row manually after inspecting confidence.')

DEMO_ROWS = {subject: choose_demonstration_row(subject) for subject in manifest['subjects']}
# To inspect another frame, change a row here and re-run the following plot cells.
examples = {subject: read_pair(subject, row) for subject, row in DEMO_ROWS.items()}
display(pd.DataFrame([{'subject': k, 'radar_row': v['radar_row'],
    'native_radar_index': v['radar_sample_index'], 'pose_row': v['pose']['row'],
    'points': len(v['points']), 'people_annotation': v['pose']['people'],
    '3D_detections': len(v['pose']['xyz']), '2D_detections': len(v['pose']['xy']),
    'residual_ms': v['residual_ms']} for k, v in examples.items()]))

## 4. Point-cloud and 3D-pose plots

Both are already in the same **world frame, metres: X right, Y forward, Z up**.
The plots draw all radar points from one frame. Dark skeleton edges require finite
endpoints and Kinect confidence ≥ 2; faint crosses show finite lower-confidence joints.
That threshold is an illustrative policy, not an approved experiment specification.
Each plot states the actual row and timing residual. Visual agreement is a diagnostic,
not evidence that timing or calibration is accurate over the entire recording.

In [ ]:
edges3 = [('pelvis', 'spine - navel'), ('spine - navel', 'spine - chest'),
          ('spine - chest', 'neck'), ('neck', 'head')]
for side in ('left', 'right'):
    edges3 += [('spine - chest', f'{side} clavicle'), (f'{side} clavicle', f'{side} shoulder'),
               (f'{side} shoulder', f'{side} elbow'), (f'{side} elbow', f'{side} wrist'),
               (f'{side} wrist', f'{side} hand'), (f'{side} hand', f'{side} handtip'),
               (f'{side} hand', f'{side} thumb'), ('pelvis', f'{side} hip'),
               (f'{side} hip', f'{side} knee'), (f'{side} knee', f'{side} ankle'),
               (f'{side} ankle', f'{side} foot')]
edge_indices3 = [(joint_names3.index(a), joint_names3.index(b)) for a, b in edges3]

def equal_3d_limits(ax, xyz):
    xyz = xyz[np.isfinite(xyz).all(axis=1)]
    low, high = xyz.min(axis=0), xyz.max(axis=0)
    centre = (low + high) / 2
    radius = max(float((high-low).max()/2), .5) * 1.05
    ax.set_xlim(centre[0]-radius, centre[0]+radius)
    ax.set_ylim(centre[1]-radius, centre[1]+radius)
    ax.set_zlim(centre[2]-radius, centre[2]+radius)
    ax.set_box_aspect((1, 1, 1))

fig = plt.figure(figsize=(14, 6), layout='constrained')
for col, (subject, sample) in enumerate(examples.items(), 1):
    ax = fig.add_subplot(1, len(examples), col, projection='3d')
    pts, pose = sample['points'], sample['pose']
    artist = ax.scatter(*pts[:, :3].T, c=pts[:, 3], cmap='coolwarm', vmin=-2, vmax=2,
                        s=5, alpha=.5, label='radar points')
    for person, xyz in enumerate(pose['xyz']):
        finite = pose['finite3d'][person]
        valid = finite & (pose['confidence3d'][person] >= 2)
        ax.scatter(*xyz[valid].T, color='#183153', s=22)
        ax.scatter(*xyz[finite & ~valid].T, color='#949ba5', s=18, marker='x')
        for a, b in edge_indices3:
            if valid[a] and valid[b]:
                ax.plot(*xyz[[a, b]].T, color='#183153', linewidth=2)
    equal_3d_limits(ax, np.concatenate([pts[:, :3], pose['xyz'].reshape(-1, 3)]))
    ax.set(xlabel='World X / m', ylabel='World Y / m', zlabel='World Z / m',
           title=f"{subject} | radar row {sample['radar_row']}\nKinect − radar: {sample['residual_ms']:+.2f} ms")
    ax.view_init(elev=18, azim=-65)
fig.colorbar(artist, ax=fig.axes, shrink=.65, label='Radial velocity / m s⁻¹ (colour clipped to ±2)')
plt.show()

The full-scene view above includes distant points and outliers. This second view
uses a **display-only** 3 m cube around the finite skeleton to make limb geometry
readable. It reports how many points fall inside the view; nothing is removed from
the loaded sample. Solid edges have confidence ≥ 2 at both ends; dashed grey edges
have a finite lower-confidence endpoint. This pose-centred view is a diagnostic,
not an input crop suitable for deployment without labels.

In [ ]:
fig = plt.figure(figsize=(13, 6), layout='constrained')
for col, (subject, sample) in enumerate(examples.items(), 1):
    ax = fig.add_subplot(1, len(examples), col, projection='3d')
    pts, pose = sample['points'], sample['pose']
    finite_xyz = pose['xyz'][pose['finite3d']]
    centre = (finite_xyz.min(axis=0) + finite_xyz.max(axis=0)) / 2
    half_width = 1.5
    inside = (np.abs(pts[:, :3] - centre) <= half_width).all(axis=1)
    shown = pts[inside]
    artist = ax.scatter(*shown[:, :3].T, c=shown[:, 3], cmap='coolwarm', vmin=-2, vmax=2,
                        s=11, alpha=.65)
    for person, xyz in enumerate(pose['xyz']):
        finite = pose['finite3d'][person]
        valid = finite & (pose['confidence3d'][person] >= 2)
        ax.scatter(*xyz[valid].T, color='#183153', s=28)
        ax.scatter(*xyz[finite & ~valid].T, color='#949ba5', s=18, marker='x')
        for a, b in edge_indices3:
            if finite[a] and finite[b]:
                high = valid[a] and valid[b]
                ax.plot(*xyz[[a, b]].T, color='#183153' if high else '#949ba5',
                        linewidth=2 if high else 1, linestyle='-' if high else '--')
    ax.set_xlim(centre[0]-half_width, centre[0]+half_width)
    ax.set_ylim(centre[1]-half_width, centre[1]+half_width)
    ax.set_zlim(centre[2]-half_width, centre[2]+half_width)
    ax.set_box_aspect((1, 1, 1))
    ax.set(xlabel='World X / m', ylabel='World Y / m', zlabel='World Z / m',
           title=f'{subject} | display-only detail\n{int(inside.sum())} / {len(pts)} radar points inside view')
    ax.view_init(elev=16, azim=-65)
fig.colorbar(artist, ax=fig.axes, shrink=.65, label='Radial velocity / m s⁻¹')
plt.show()

## 5. 2D poses: measured image canvas, independent detections

The 26-joint 2D arrays are RTMPose/Body8-Halpe26 detections; they are **not Kinect32
projected into an image**. Preserve detector scores separately from Kinect confidence.
The ETL pipeline already filtered detections using its confidence/count policy;
this export cannot recover discarded detections or original bounding-box coordinates.

Additional source inspection on **2026-10-08** established that the MMPose overlay
videos for **both** subjects are **1280 × 720, 15 fps**. Three sampled exported
single-person poses per subject exactly matched original prediction JSON coordinates
(maximum difference 0 pixels); one middle sample per subject was visually overlaid
on the saved visualization to check that coordinate scale. The evidence snapshot below
records the source paths, JSON hashes and samples. Those original files are not needed
to run this notebook, and were not added to the export.

Use this measured canvas for these recordings. X increases right, Y down; predictions
outside the canvas are possible. Internal neural-network crop/resize dimensions are
not the coordinate system of these saved predictions. This evidence does **not**
establish the capture-mode camera matrix or a Kinect3D→colour projection.
The factory calibration's 4096×3072 sensor dimensions must not be substituted here.

In [ ]:
POSE2D_IMAGE_EVIDENCE = {'09_SY': {'video_ref': 'D:\\smart_cup_recordings\\LivingLabPose\\Body8-Halpe26 RTMPose-l_defaults\\09_SY\\Session-2023-December-04 15-44-03-011950\\visualizations\\kinect_camera_recording.mkv', 'video_metadata': {'width_px': 1280, 'height_px': 720, 'frame_count': 47205, 'fps': 15.0}, 'prediction_ref': 'D:\\smart_cup_recordings\\LivingLabPose\\Body8-Halpe26 RTMPose-l_defaults\\09_SY\\Session-2023-December-04 15-44-03-011950\\predictions\\kinect_camera_recording.json', 'prediction_sha256': '1a47be1480ee4cf3ea208ede7eb70184ee2980f9b548df99c5921f881cf065f0', 'coordinate_comparisons': [{'prediction_frame_id': 146, 'export_pose_row': 146, 'max_coordinate_difference_px': 0.0}, {'prediction_frame_id': 18855, 'export_pose_row': 18855, 'max_coordinate_difference_px': 0.0}, {'prediction_frame_id': 41740, 'export_pose_row': 41740, 'max_coordinate_difference_px': 0.0}], 'inspection_date': '2026-10-08'}, '19_MM': {'video_ref': 'D:\\smart_cup_recordings\\LivingLabPose\\Body8-Halpe26 RTMPose-l_defaults\\19_MM\\Session-2024-January-15 09-47-41-274452\\visualizations\\kinect_camera_recording.mkv', 'video_metadata': {'width_px': 1280, 'height_px': 720, 'frame_count': 34286, 'fps': 15.0}, 'prediction_ref': 'D:\\smart_cup_recordings\\LivingLabPose\\Body8-Halpe26 RTMPose-l_defaults\\19_MM\\Session-2024-January-15 09-47-41-274452\\predictions\\kinect_camera_recording.json', 'prediction_sha256': 'd2b6f782b46aeb757ccde3a0a534a04ae200808bbdad2607e488d2ff222f0489', 'coordinate_comparisons': [{'prediction_frame_id': 83, 'export_pose_row': 83, 'max_coordinate_difference_px': 0.0}, {'prediction_frame_id': 10715, 'export_pose_row': 10715, 'max_coordinate_difference_px': 0.0}, {'prediction_frame_id': 33991, 'export_pose_row': 33991, 'max_coordinate_difference_px': 0.0}], 'inspection_date': '2026-10-08'}}

display(pd.DataFrame([{'subject': subject, **entry['video_metadata'],
                      'checked_frames': [x['prediction_frame_id'] for x in entry['coordinate_comparisons']],
                      'max_difference_px': max(x['max_coordinate_difference_px'] for x in entry['coordinate_comparisons'])}
                     for subject, entry in POSE2D_IMAGE_EVIDENCE.items()]))
edges2 = [('Nose', 'L_Eye'), ('Nose', 'R_Eye'), ('L_Eye', 'L_Ear'), ('R_Eye', 'R_Ear'),
          ('Head_Apex', 'Neck'), ('Neck', 'Hip_Center'), ('L_Shoulder', 'R_Shoulder'),
          ('L_Hip', 'R_Hip')]
for side in ('L', 'R'):
    edges2 += [('Neck', f'{side}_Shoulder'), (f'{side}_Shoulder', f'{side}_Elbow'),
               (f'{side}_Elbow', f'{side}_Wrist'), ('Hip_Center', f'{side}_Hip'),
               (f'{side}_Hip', f'{side}_Knee'), (f'{side}_Knee', f'{side}_Ankle'),
               (f'{side}_Ankle', f'{side}_Heel'), (f'{side}_Heel', f'{side}_BigToe'),
               (f'{side}_Heel', f'{side}_SmallToe')]
edge_indices2 = [(joint_names2.index(a), joint_names2.index(b)) for a, b in edges2]
fig, axes = plt.subplots(1, len(examples), figsize=(14, 4.8), layout='constrained', squeeze=False)
for ax, (subject, sample) in zip(axes[0], examples.items()):
    pose = sample['pose']
    for person, xy in enumerate(pose['xy']):
        valid = pose['finite2d'][person] & (pose['confidence2d'][person] >= .5)
        for a, b in edge_indices2:
            if valid[a] and valid[b]:
                ax.plot(*xy[[a, b]].T, color='#7b8694', linewidth=1.5)
        scatter = ax.scatter(*xy[valid].T, c=pose['confidence2d'][person][valid],
                             cmap='viridis', vmin=0, vmax=1, s=30)
    meta = POSE2D_IMAGE_EVIDENCE[subject]['video_metadata']
    ax.set(xlim=(0, meta['width_px']), ylim=(meta['height_px'], 0), aspect='equal',
           xlabel='Image X / px', ylabel='Image Y / px',
           title=f"{subject} | Kinect row {pose['row']} | 1280 × 720 canvas")
    ax.grid(alpha=.15)
fig.colorbar(scatter, ax=axes.ravel().tolist(), shrink=.65, label='2D keypoint score (plot threshold 0.5)')
plt.show()

## 6. Timing, gaps and correspondence

The selected mapping is `initial_rgb_to_raw_v001` for both subjects: `identity_time`
with radar timing interpolated from recording start/end and Kinect wallclock from
PTS. A small nearest-neighbour residual only measures agreement **under this model**;
it does not measure the true acquisition lag. Do not estimate synchronization quality
from the histogram alone. Review motion alignment across the recording before learning.

The exporter picks nearest predicted Kinect time, earlier frame on a tie, within
the selected tolerance (75 ms here), no extrapolation and no cross-block matching.
Rejected candidates retain `nearest_residual_ms`; only nonnegative `pose_row` is usable.
There is no interpolation of joint coordinates and no fixed radar↔Kinect rate ratio.

`timeline_time_s + timeline_origin` reconstructs a modality's selected timeline.
`aligned_time_s + aligned_origin` reconstructs the common radar time coordinate.
Naive wallclock origins are UTC-like numeric coordinates, not verified UTC timestamps.
RGB `rgb_pts_s` is for locating source-video content; dividing an index by nominal fps
is not a substitute for these exported times.

`sequence_block` marks cadence gaps over three nominal periods and missing native
indices in either modality. A run may have **many** blocks. Never bridge them merely
because a radar file is continuous. Unprocessed frames and missing labels also need
explicit policy in the window assembler; preserve timestamps when selecting data.

In [ ]:
timing_rows = []
fig, axes = plt.subplots(len(segments), 2, figsize=(13, 6), layout='constrained', squeeze=False)
for axes_row, subject in zip(axes, segments):
    with open_pair(subject) as (r, p):
        t = r['frames/aligned_time_s'][:]
        pt = p['frames/aligned_time_s'][:]
        matched = r['correspondence/matched'][:]
        residual = r['correspondence/nearest_residual_ms'][:]
        blocks = r['frames/sequence_block'][:]
        tolerance = float(r['correspondence'].attrs['tolerance_ms'])
        axes_row[0].hist(residual[matched], bins=50, color='#287da6')
        axes_row[0].axvline(-tolerance, color='#c65b38', linestyle='--')
        axes_row[0].axvline(tolerance, color='#c65b38', linestyle='--')
        axes_row[0].set(title=f'{subject}: accepted model residuals', xlabel='Kinect − radar / ms', ylabel='Radar frames')
        step = max(1, len(t)//3000)  # Display thinning only; statistics use every frame.
        axes_row[1].plot(t[::step]/60, blocks[::step], linewidth=1, color='#287da6')
        axes_row[1].set(title=f'{subject}: shared continuity blocks', xlabel='Aligned time / minutes', ylabel='Block ID')
        timing_rows.append({'subject': subject, 'radar_nominal_period_s': float(r.attrs['nominal_sample_period_s']),
            'radar_median_dt_s': float(np.median(np.diff(t))),
            'Kinect_median_dt_s': float(np.median(np.diff(pt))),
            'blocks': len(np.unique(blocks)), 'matched': int(matched.sum()),
            'unmatched': int((~matched).sum()), 'span_s': float(t[-1]-t[0])})
plt.show()
display(pd.DataFrame(timing_rows))
display(pd.DataFrame(manifest['coverage'])[['subject_id','modality','total_frames','exported_frames','excluded_sample_ranges']])

## 7. People, confidence, missing data and activity annotations

`num_people` is imported annotated presence. `num_2d`/`num_3d` are detector summary
counts; `pose2d_person_count`/`pose_person_count` describe actual stored arrays.
Do not silently choose person 0 when two people are present, even if only one is detected.
A conservative single-person example policy requires annotation == 1 **and** exactly
one stored 3D person, usable root, finite coordinates and an explicit confidence mask.
This still does not establish identity across time. Multi-person training needs an
association design and appropriate model/target contracts.

Kinect confidence uses native levels (these files contain 0, 1, 2); use a documented
threshold, not a probability interpretation. RTMPose keypoint and bounding-box scores
are separate quantities. An empty available 3D payload can mean no detected bodies;
it is different from an absent payload. For canonical `Pose3DTarget`, **all coordinates
must be finite even where validity is false**: replace invalid values in a copy with a
finite placeholder, retain a false mask, and require a valid pelvis for pelvis-relative
targets. Do not alter the source arrays.

Activity JSON preserves arbitrary tiers. Current keys include `01-Activity`, `01-Notes`,
`01-Objects`, `01-PhysicalState`, `02-Activity`, `02-Objects`, `02-PhysicalState`.
Empty lists mean no label in that tier, not an automatically named negative class.
Tier prefixes are not pose-array indices or tracker IDs. Activity annotations are
metadata, not the framework's pose target kinds. Keep their labels out of model inputs
unless an experiment explicitly permits them.

In [ ]:
quality_rows = []
for subject in segments:
    with open_pair(subject) as (r, p):
        pr = r['correspondence/pose_row'][:]
        safe = pr.clip(0)
        base = ((pr >= 0) & r['frames/payload_available'][:] & (r['frames/point_count'][:] > 0))
        one = base & (p['frames/num_people'][:][safe] == 1) & (p['frames/pose_person_count'][:][safe] == 1)
        counts = p['frames/num_people'][:]
        quality_rows.append({'subject': subject, 'all_radar_rows': len(pr),
            'available_nonempty_with_match': int(base.sum()), 'single_annotated_and_detected_3D': int(one.sum()),
            'Kinect_zero_people': int((counts == 0).sum()), 'Kinect_two_or_more': int((counts >= 2).sum()),
            'Kinect_missing_3D_payload': int((~p['frames/payload_available'][:]).sum()),
            'Kinect_missing_2D_payload': int((~p['frames/pose2d_payload_available'][:]).sum()),
            '3D_confidence_levels': np.unique(p['poses/confidence'][:]).tolist()})
display(pd.DataFrame(quality_rows))
print('Frame-level screening only: these are NOT accepted temporal training-window counts.')
for subject, sample in examples.items():
    print(subject, '| RGB PTS:', sample['pose']['rgb_pts_s'], '| activity:', sample['pose']['activity'])

## 8. Calibration and coordinate conversion

Three different calibration concepts are present:

| Kind | Where | What it establishes |
|---|---|---|
| Offline radar processing | `selections[].cloud.provenance_json` and `exact_processing_recipe` | Generator/configuration and the per-acquisition radar calibration inputs; these inputs differ by subject. |
| Kinect↔radar spatial alignment | `geometry.spatial_calibration` | Selected `desk_all.json` with rigid matrices and full fitting provenance; assumes the sensor mounting applies to both recordings. |
| Kinect factory camera calibration | `selections[].kinect.camera_calibration` | Original intrinsics/distortion/extrinsics document. Its sensor dimensions are not automatically the capture-mode or prediction-image dimensions. |

The shared `processing_compatibility_id` says the processing method/settings and
calibration procedure are compatible. It does **not** say the exact calibration
inputs/results are identical. Keep both per-run exact recipe IDs and source hashes.

The export applied Kinect native mm→radar metres with the selected spatial calibration,
then the same radar→world transform to both modalities. Floor pitch (30°) and sensor
height (1.76 m) remain GUI assumptions. For column vectors:
`world = radar_rotation @ radar_sensor + translation_m`.
For row-vector NumPy arrays, the inverse below is `(world - translation) @ rotation`.
**Do not run the historical Living Lab replay conversion again** on these world arrays.
Doppler remains measured sensor radial velocity; a rigid XYZ rotation does not turn
it into a 3D velocity vector, and its sign must not be flipped to imitate another dataset.

Only the declared inverse world transform is needed to put both modalities into
`radar_link`. Do not subtract Kinect's camera origin from radar-aligned targets.
No 3D→2D projection is attempted here: capture-mode intrinsics/distortion, depth-to-colour
transform conventions, and source crop/resize history still need verification.

In [ ]:
geometry = manifest['geometry']
R = np.asarray(geometry['radar_rotation'], dtype=np.float64)
t = np.asarray(geometry['translation_m'], dtype=np.float64)
assert np.allclose(R.T @ R, np.eye(3), atol=1e-8)
assert np.isclose(np.linalg.det(R), 1)

def world_to_radar(xyz):
    return (np.asarray(xyz, dtype=np.float64) - t) @ R

def radar_to_world(xyz):
    return np.asarray(xyz, dtype=np.float64) @ R.T + t

sensor_examples = {}
for subject, sample in examples.items():
    points = sample['points'].copy()
    points[:, :3] = world_to_radar(points[:, :3])
    xyz = world_to_radar(sample['pose']['xyz'])
    np.testing.assert_allclose(radar_to_world(points[:, :3]), sample['points'][:, :3], atol=2e-6)
    np.testing.assert_allclose(points[:, 3:], sample['points'][:, 3:])
    sensor_examples[subject] = {'points': points, 'xyz': xyz}

calibration_rows = []
for selection in manifest['selections']:
    camera = selection['kinect']['camera_calibration']
    exact = selection['exact_processing_recipe']
    calibration_rows.append({'subject': selection['selection']['subject_id'],
        'exact_recipe_id': selection['exact_processing_recipe_id'],
        'radar_calibration_frame_hashes': len(exact.get('calibration_sha256', [])),
        'camera_calibration_sha256': camera['sha256'] if camera else None})
display(pd.DataFrame(calibration_rows))
print('Shared compatibility:', manifest['processing_compatibility_id'])
print('Spatial calibration:', geometry['spatial_calibration']['source_name'], geometry['spatial_calibration']['sha256'])
print('World→radar round trips passed; nonspatial point columns are unchanged.')

## 9. Prepare a finite single-person target without claiming a new skeleton

This example keeps all **32 native Kinect joints** and demonstrates an explicit
confidence policy (≥ 2) plus validity. The pelvis must pass. It is a target-construction
example, not a complete approved selection policy. A temporal policy must evaluate
all required targets/people and frames across its window.

The existing MM-Fi model uses a different 17-joint schema. **Do not take the first 17
Kinect joints**, and do not align 2D and 3D arrays by joint index. Use named, reviewed
correspondences. Candidate pelvis→pelvis_center and spine-chest→thorax definitions
still need an anatomical convention; MM-Fi `face_center` and `head` are explicitly
provisional in the framework. Either train a native Kinect32 output with compatible
model/loss/metric/bundle support, or implement a versioned, reviewed Kinect32→target
mapping with unsupported joints invalid. A 32-joint target is valid at the core data
boundary; that does not make it compatible with an existing 17-joint checkpoint.

In [ ]:
MIN_KINECT_CONFIDENCE = 2  # Demonstration setting: record any experimental replacement.

def prepare_native_target(sample):
    p = sample['pose']
    if p is None or p['people'] != 1 or not p['available3d'] or len(p['xyz']) != 1:
        return None
    xyz = world_to_radar(p['xyz'][0])
    valid = (p['finite3d'][0] & np.isfinite(xyz).all(axis=1)
             & np.isfinite(p['confidence3d'][0]) & (p['confidence3d'][0] >= MIN_KINECT_CONFIDENCE))
    root_index = joint_names3.index('pelvis')
    if not valid[root_index]:
        return None
    safe_xyz = np.where(valid[:, None], xyz, 0.0).astype(np.float32)
    assert np.isfinite(safe_xyz).all()
    return {'joints': safe_xyz, 'valid_mask': valid, 'coordinate_frame': 'radar_link',
            'reference': 'sensor_frame', 'units': 'm', 'root_index': root_index}

target_examples = {s: prepare_native_target(x) for s, x in examples.items()}
assert all(v is not None for v in target_examples.values())
display(pd.DataFrame([{'subject': s, 'shape': v['joints'].shape,
                      'valid_joints': int(v['valid_mask'].sum()), 'root_valid': bool(v['valid_mask'][v['root_index']])}
                     for s, v in target_examples.items()]))
# Pelvis-relative encoding changes the pose reference, not radar geometry:
native_target = target_examples[manifest['subjects'][0]]
relative_xyz = native_target['joints'] - native_target['joints'][native_target['root_index']]
assert np.allclose(relative_xyz[native_target['root_index']], 0)
print('Relative target keeps the same validity mask; its coordinates remain finite.')

## 10. Build the ML sequence and frame catalogues

In `vip_hpe_ml/datasets/index.py`, `SampleIndex` requires records for each sequence
to be contiguous in the catalogue, with **framework `frame_index` = 0,1,2,…** and
strictly increasing timestamps. Native sample indices must instead remain in
`source_locator`/metadata. This local renumbering must not invent time or hide gaps.

The example below retains available radar frames, including valid empty clouds.
It splits at every exported continuity block and every hole caused by an unavailable
radar payload. Each span gets a unique sequence ID. It stores aligned timestamps
and also sequence-relative timestamps; origin is recorded per sequence. Pose quality
is not used to erase frames: `load_target` may return None, and supervised window
selection can explicitly reject unusable targets. Do not join disjoint accepted frames
into an apparently contiguous motion sequence.

Group every span from the same subject together for splitting. A sequence is not an
independent subject. With only two subjects you cannot make nonempty disjoint
train/validation/test subject groups. Choose a declared two-subject development
protocol or add subjects; never disguise random overlapping-window splits as held-out
subject evaluation. Fit bounds, normalization and tuning decisions only on training data.

In [ ]:
def build_catalogues():
    frame_tables, sequence_rows = [], []
    for subject, s in segments.items():
        with open_pair(subject) as (r, p):
            available = r['frames/payload_available'][:]
            keep = np.flatnonzero(available)
            block = r['frames/sequence_block'][:]
            native = r['frames/sample_index'][:]
            times = r['frames/aligned_time_s'][:]
            refs = r['correspondence/pose_row'][:]
            period = float(r.attrs['nominal_sample_period_s'])
        if not len(keep):
            continue
        cuts = np.flatnonzero((np.diff(keep) != 1) | (np.diff(block[keep]) != 0)
                              | (np.diff(native[keep]) != 1)) + 1
        for span, rows in enumerate(np.split(keep, cuts)):
            sequence_id = f"{s['segment_id']}:block{block[rows[0]]}:span{span}"
            origin = float(times[rows[0]])
            sequence_rows.append(dict(sequence_id=sequence_id, subject=subject, session=s['radar_run_id'],
                segment_id=s['segment_id'], block=int(block[rows[0]]), aligned_origin_s=origin,
                nominal_sample_period_s=period, frame_count=len(rows)))
            frame_tables.append(pd.DataFrame(dict(subject=subject, sequence_id=sequence_id,
                frame_index=np.arange(len(rows)), radar_row=rows, native_sample_index=native[rows],
                timestamp_s=times[rows]-origin, aligned_time_s=times[rows], pose_row=refs[rows])))
    return pd.DataFrame(sequence_rows), pd.concat(frame_tables, ignore_index=True)

sequence_catalogue, frame_catalogue = build_catalogues()
for _, group in frame_catalogue.groupby('sequence_id', sort=False):
    assert np.array_equal(group.frame_index.to_numpy(), np.arange(len(group)))
    assert np.all(np.diff(group.timestamp_s) > 0)
display(sequence_catalogue.groupby('subject').agg(sequences=('sequence_id','size'), radar_frames=('frame_count','sum')))
display(frame_catalogue.head())
print('All framework indices are contiguous; native IDs and physical timestamps are retained separately.')

## 11. The exact adapter boundary and optional constructor checks

Implement `LivingLabAdapter` in the ML repository, not by modifying these HDF5 files.
The current protocol in `ml/vip_hpe_ml/datasets/base.py` requires:

| Contract | How to populate it |
|---|---|
| `descriptor: DatasetDescriptor` | Unique dataset ID, `syncwb.training_export.v3`, immutable export fingerprint, root identifier, point/target schema IDs, calibration/mapping provenance. Also record your adapter/selection/coordinate policy version; an unchanged data hash does not identify a changed adapter. |
| `sample_index: SampleIndex` | `SequenceRecord` + `AtomicSampleRecord`, ordered as above. `group_ids` includes `subject` and optionally session. Source locators contain relative HDF5 paths, row and native ID. |
| `load_frame(sample) -> RadarFrameSample` | Float32 finite radar-frame points with explicit schema. Invert world→radar first; preserve Doppler and SNR. Never append association IDs as an anonymous sixth feature. |
| `available_target_kinds(sample)` | Return only kinds usable under your documented policy. A supported match alone does not make a target usable. |
| `load_target(sample, kind) -> PoseTarget or None` | Follow pose_row, select people explicitly, apply confidence/finite masks, use declared skeleton and units. Return None for unavailable or policy-rejected supervision. |

For the current IWR6843 contract, use XYZ/doppler/signal_strength (5 columns) in
`radar_link`; retain the exported SNR's **linear power ratio** unit in provenance.
Its meaning is SNR, not MM-Fi intensity. The existing `IWR6843_POINT_SCHEMA` declares
`source_native` signal units; a more precise new schema requires registry/deployment
support. The baseline currently sets `signal_mode: none`; robust/rank transforms do
not prove equivalence between sensors. Keep raw association codes separately, and
leave `point_track_ids=None`/association unavailable until their mapping is verified.
There are no tracker centroids/velocities in this export; use an appropriate point-based
anchor rather than fabricating tracker observations.

The constructor example below is optional. Set `FRAMEWORK_ROOT` to your checkout if
you want to exercise the real classes. The rest of this notebook has no dependency
on that checkout. It checks one finite **native Kinect32** sample, not a trained model
or a finished adapter. It marks raw radar time as synthesized because this release's
radar timeline is an interpolation from recording endpoints, while retaining its source
model as provenance. Do not label it a measured per-frame sensor clock.

In [ ]:
FRAMEWORK_ROOT = None  # e.g. Path(r'C:\path\to\VIP_mmWave_HPE_demo')
if FRAMEWORK_ROOT is None:
    print('Optional framework constructor checks skipped. Set FRAMEWORK_ROOT to your checkout to run them.')
else:
    import sys
    checkout = Path(FRAMEWORK_ROOT).resolve()
    sys.path[:0] = [str(checkout / 'src' / 'vip_hpe_core'), str(checkout / 'ml')]
    from vip_hpe_core.data import RadarFrameSample, Pose3DTarget, PoseReference, TimestampSource
    from vip_hpe_core.schemas.points import IWR6843_POINT_SCHEMA
    from vip_hpe_core.schemas.joints import AZURE_KINECT_32_SCHEMA
    from vip_hpe_ml.datasets.base import DatasetDescriptor, PoseTargetKind
    from vip_hpe_ml.datasets.index import SequenceRecord, AtomicSampleRecord, SampleIndex
    from vip_hpe_ml.targets import PelvisRelativeTargetStrategy

    descriptor = DatasetDescriptor(dataset_id='living_lab_09_SY_19_MM',
        export_format_version=manifest['schema'], dataset_fingerprint=manifest['dataset_fingerprint'],
        root_identifier=DATA_ROOT.name, point_schema_id=IWR6843_POINT_SCHEMA.schema_id,
        target_schema_ids={PoseTargetKind.POSE_3D: AZURE_KINECT_32_SCHEMA.schema_id},
        provenance={'adapter_policy': 'notebook_native32_demo_v1', 'source_signal_unit': 'linear_power_ratio',
                    'geometry_profile': geometry['profile_id'], 'sync_release': manifest['synchronization_release_id']})
    seq_records = tuple(SequenceRecord(row.sequence_id, row.nominal_sample_period_s,
        group_ids={'subject': row.subject, 'session': row.session},
        metadata={'segment_id': row.segment_id, 'block': row.block, 'aligned_origin_s': row.aligned_origin_s})
        for row in sequence_catalogue.itertuples(index=False))
    atom_records = tuple(AtomicSampleRecord(
        sample_id=f'{row.sequence_id}:f{row.frame_index}', sequence_id=row.sequence_id,
        frame_index=int(row.frame_index), timestamp_s=float(row.timestamp_s),
        source_locator={'radar_path': segments[row.subject]['files']['radar']['path'],
                        'radar_row': int(row.radar_row), 'native_sample_index': int(row.native_sample_index)},
        metadata={'pose_row': int(row.pose_row), 'subject': row.subject})
        for row in frame_catalogue.itertuples(index=False))
    sample_index = SampleIndex(descriptor.dataset_id, descriptor.dataset_fingerprint, seq_records, atom_records)
    subject = manifest['subjects'][0]
    position = int(np.flatnonzero((frame_catalogue.subject.to_numpy() == subject)
                                  & (frame_catalogue.radar_row.to_numpy() == DEMO_ROWS[subject]))[0])
    record = sample_index.records[position]
    sample = examples[subject]
    radar_frame = RadarFrameSample(sample_id=record.sample_id, sequence_id=record.sequence_id,
        frame_index=record.frame_index, timestamp_s=record.timestamp_s, timestamp_source=TimestampSource.SYNTHESIZED,
        points=sensor_examples[subject]['points'][:, :5].astype(np.float32), point_schema=IWR6843_POINT_SCHEMA,
        metadata={'raw_association_codes': sample['points'][:, 5].copy(),
                  'timing_basis': 'radar_raw_wallclock_from_start_end', 'snr_unit': 'linear_power_ratio'})
    prepared = target_examples[subject]
    assert tuple(joint_names3) == AZURE_KINECT_32_SCHEMA.joint_names
    target = Pose3DTarget(joints=prepared['joints'], valid_mask=prepared['valid_mask'],
        joint_schema=AZURE_KINECT_32_SCHEMA, coordinate_frame='radar_link', reference=PoseReference.SENSOR_FRAME)
    encoded = PelvisRelativeTargetStrategy().encode(target)
    print('Core contracts passed:', radar_frame.points.shape, target.joints.shape, encoded.values.shape)
    print('ML SampleIndex validated:', len(sample_index.sequences), 'sequences,', len(sample_index.records), 'frames')

## 12. Temporal windows, model integration and evaluation

The adapter is the data boundary, not the whole experiment. The inspected framework's
standard experiment currently builds `PackedMMFiAdapter` and its audit through the
`mmfi_packed` factory and uses `MMFICrossEnvironmentSplitPolicy`. Changing a dataset
path in the MM-Fi YAML will **not** load these files. Add a Living Lab factory/audit and
subject split configuration, reusing generic index/window/training code where possible.

**Windows:** `TemporalWindowSpec` requests physical time slots. The current MM-Fi
baseline is 5 centered slots spaced 0.1 s with 0.049 s maximum slot error. The radar
here is nominally 0.05 s/frame; blindly taking five adjacent frames changes the
physical span from 0.4 s to 0.2 s. Deliberately choose between preserving a model's
physical-time contract and training a new cadence. Radar-slot matching tolerance is
distinct from the export's 75 ms radar↔pose tolerance. For size > 1 the core requires
slot error < half the slot period. Centered windows need future data; a causal deployment
must use a compatible latest-aligned design. Respect the sequence spans above, preserve
missing slots, and use the framework's `OfflineWindowIndexer`/`TemporalWindowAssembler`.

**Preprocessing/targets:** `HPEWindowDataset` currently requires anchor-relative point
features and pelvis-relative targets. The radar anchor and anatomical pelvis are different
quantities. Keep radar/pose sensor geometry consistent, choose a valid root, and let the
target strategy perform root subtraction once. Do not use a label-derived translation
to secretly align radar input. Crop coordinates must be in `radar_link`, not world metres.
The baseline 3 m capture region is a model/deployment choice, not guaranteed Living Lab
coverage. Inspect rejection rates. Label-assisted virtual crops are a supervised protocol;
they cannot be treated as deployable input selection when labels are unavailable.

**2D:** the adapter protocol admits `Pose2DTarget`, but the current supervised trainer is
3D. Supporting a Halpe26 2D schema, valid image coordinates, a suitable loss/model and
evaluation is separate work. Returning 2D targets does not add a 2D training path. Do not
claim the 2D detections supervise the same person as Kinect3D without association.

**Evaluation:** declare a reviewed joint mapping and mask policy, valid pelvis policy,
target frame/reference, supported-person policy, temporal tolerances and crop protocol.
Report rejection reasons/coverage and valid-joint counts alongside error metrics. Retain
original row IDs for every accepted example. Repeated pose labels and overlapping windows
must not cross splits. Do not fit SimCC bounds on validation/test targets.

**Fine-tuning/deployment:** create a new experiment with checkpoint provenance. `resume`
is for the same saved run identity, not switching datasets. Revisit output joint count,
decoder/bounds, metrics and exported bundle schema when changing topology; no weights-only
compatibility is promised. `load_target=None` is permitted by the protocol but does not
implement an unsupervised training recipe.

The executable example below requests **5 slots at 0.1 s spacing** within the
demonstration frame's sequence. Its illustrative 0.024 s radar-slot tolerance is
stricter than the MM-Fi baseline and is unrelated to the export's 75 ms pose-match
tolerance. A slot outside the sequence or too far from its nearest radar frame is
reported missing; it is never filled from the next sequence. In the finished
adapter, use the framework's window indexer/assembler rather than maintaining a
second subtly different windowing implementation.

In [ ]:
WINDOW_SIZE, SLOT_PERIOD_S, SLOT_TOLERANCE_S = 5, .1, .024
assert SLOT_TOLERANCE_S < SLOT_PERIOD_S / 2
window_examples = []
for subject, demo in examples.items():
    centre_record = frame_catalogue.loc[(frame_catalogue.subject == subject)
                                      & (frame_catalogue.radar_row == demo['radar_row'])].iloc[0]
    sequence = frame_catalogue.loc[frame_catalogue.sequence_id == centre_record.sequence_id]
    times = sequence.timestamp_s.to_numpy()
    relative_slots = (np.arange(WINDOW_SIZE) - WINDOW_SIZE // 2) * SLOT_PERIOD_S
    desired = float(centre_record.timestamp_s) + relative_slots
    right = np.searchsorted(times, desired).clip(0, len(times)-1)
    left = (right-1).clip(0)
    nearest = np.where(np.abs(times[left]-desired) <= np.abs(times[right]-desired), left, right)
    error = times[nearest]-desired
    valid = ((np.abs(error) <= SLOT_TOLERANCE_S)
             & (desired >= times[0]) & (desired <= times[-1]))
    assert len(np.unique(nearest[valid])) == int(valid.sum())
    for slot in range(WINDOW_SIZE):
        record = sequence.iloc[int(nearest[slot])]
        window_examples.append({'subject': subject, 'slot': slot,
            'requested_relative_s': relative_slots[slot],
            'actual_relative_s': float(record.timestamp_s-centre_record.timestamp_s) if valid[slot] else np.nan,
            'radar_slot_error_ms': 1000*error[slot], 'available': bool(valid[slot]),
            'radar_row': int(record.radar_row) if valid[slot] else -1,
            'native_sample_index': int(record.native_sample_index) if valid[slot] else -1})
display(pd.DataFrame(window_examples))
print('A 5-slot window spans', (WINDOW_SIZE-1)*SLOT_PERIOD_S, 'seconds; it does not mean five adjacent radar rows.')

## 13. Provenance map and files students should read

`manifest.json` is the completion/provenance record. Its fields are:

| Fields | Meaning |
|---|---|
| `schema`, `subjects`, `created_at` | Format, membership and export timestamp. |
| `dataset_fingerprint` | SHA-256 of manifest content before adding itself and created_at; includes payload hashes. |
| `processing_compatibility_id`, `processing_compatibility` | Shared processing method; per-subject calibration sources may differ. |
| `mapping_profile_id`, `mapping_profile`, `synchronization_release_id` | Shared timing policy and identity of exact per-run sync snapshots. |
| `geometry` | Axes, units, world matrices/assumptions and full spatial calibration document/hash. |
| `scope`, `timestamp_policy`, `missing_policy`, `person_policy`, `pose2d_policy` | Interpretation and missing/person/prediction rules. |
| `coverage` | Original vs exported frame counts, excluded native-index ranges and reasons. |
| `segments` | Subject/run/segment IDs, radar/pose counts, unmatched count, aligned origin, sequence boundaries, file paths/hashes, radar/3D missing-payload counts. |
| `source_artifacts` | Consumed NPZ/JSONL relative references and SHA-256; provenance only, not needed to read this package. |
| `exporter_sha256`, `geometry_implementation_sha256` | Source implementation hashes used at export time. |
| `selections` | Exact source records and calibration, detailed below. |

Each selection has `selection` (subject/mapping/cloud IDs), `mapping`, `sync_model`,
`model_anchors` (canonical records), `cloud` (registry record with JSON-string
`provenance_json`), `exact_processing_recipe_id`, `exact_processing_recipe`,
`timelines` (selected model record/fingerprint for radar and pose), and `kinect`.
`kinect.video_assets` preserves external RGB references/integrity records;
`video_included` is false; `camera_calibration` contains `source_ref`, `sha256`,
`document` and `interpretation` (or null when absent). Its vendor document retains
`CalibrationInformation.Cameras`, intrinsics/model parameters, sensor dimensions,
`Rt` extrinsics, `InertialSensors` and `Metadata` unchanged. Do not guess vendor
normalization/order/units from array shape. JSON-valued canonical columns remain strings
and need an extra `json.loads`. Historical paths and hash lists are provenance, not
runtime dependencies or training feature arrays.

The following paths are relative to the `VIP_mmWave_HPE_demo` checkout inspected for
this guide on 2026-10-08. Generic code should not gain hard-coded Living Lab filenames.

| File | Responsibility |
|---|---|
| `ml/vip_hpe_ml/datasets/base.py` | Adapter and descriptor protocol. |
| `ml/vip_hpe_ml/datasets/mmfi.py` | Existing adapter example; its naming/audit are MM-Fi-specific. |
| `ml/vip_hpe_ml/datasets/index.py` | Sequence/sample invariants and fingerprints. |
| `ml/vip_hpe_ml/datasets/splits.py`, `windows.py` | Group membership and physical-time windows. |
| `ml/vip_hpe_ml/datasets/hpe_window_dataset.py`, `capture_regions.py` | Source loading, crop, augmentation, preprocessing and target assembly. |
| `src/vip_hpe_core/vip_hpe_core/data/radar.py`, `data/pose.py` | Finite immutable frame/target constructors. |
| `src/vip_hpe_core/vip_hpe_core/schemas/points.py`, `schemas/joints.py` | Versioned semantic schemas and explicit joint mapping. |
| `ml/vip_hpe_ml/targets/pose3d.py` | Pelvis-relative target encoding. |
| `src/vip_hpe_core/vip_hpe_core/temporal/spec.py`, `preprocessing/point_tokens.py` | Physical timing and feature/crop semantics. |
| `ml/vip_hpe_ml/training/standard_experiment.py` | Current MM-Fi factory/audit/split assembly that needs a Living Lab entry point. |
| `docs/ml_development.md` | Current integration gaps, fine-tuning and evaluation conventions. |

Per-worker HDF5 handles should be opened lazily and closed explicitly; do not share an
open parent-process handle with DataLoader workers. Cache small frame catalogues and
ragged offsets if helpful; avoid loading all `points/values` into RAM. Keep source
reads deterministic and leave stochastic sampling/augmentation in preprocessing.

In [ ]:
FRAMEWORK_SOURCE_SNAPSHOT = {'ml/vip_hpe_ml/datasets/base.py': '6c354db7d22b77933d52222e3b0ee840824241cfff91a34aeda4416d14a453d8', 'ml/vip_hpe_ml/datasets/index.py': '0b4c39754886767d53fcf4389070cee4169645419c8a77efe537fb07a4073680', 'ml/vip_hpe_ml/datasets/hpe_window_dataset.py': '9897cd39a8069ee08491a6b3f1b489751dcebed185c748776011bbdee8e7e736', 'ml/vip_hpe_ml/training/standard_experiment.py': '2df40b8f9f51c80853402c8a65af918f4e8c6b10b20c6e97c4eafef3ba2358cf', 'src/vip_hpe_core/vip_hpe_core/data/radar.py': '7ccb61ff33f28d339d65321e00fb4626debbc6479b1ea79b2d6d8d9e5632f69c', 'src/vip_hpe_core/vip_hpe_core/data/pose.py': '98c39f33b4cad0959e4ea81a936044a38f44a22728e6977c15ec1be717d4eed5', 'src/vip_hpe_core/vip_hpe_core/schemas/points.py': 'e91f4b2cb352e796d8b59b30449fb27b650375c533ae83203f317f94cc5c78d7', 'src/vip_hpe_core/vip_hpe_core/schemas/joints.py': 'ceba46100fb97a60cf8b7ed1d42bd05e6cfab45d749cbb1d3bff90236f78e0a9', 'src/vip_hpe_core/vip_hpe_core/temporal/spec.py': 'b51e442a2c13e65ce6b314926af38dae2329fc31d3ffbd09d51890cd98eab9f5', 'ml/vip_hpe_ml/targets/pose3d.py': '43036af4a5d660f1dc3aaffb8ec2cc75a8e32cde6bdebeeb0a94f57a2c98d41a'}

# Decode nested provenance without dumping megabytes of frame/calibration records.
selection = manifest['selections'][0]
source_cloud = json.loads(selection['cloud']['provenance_json'])
print('Cloud provenance keys:', sorted(source_cloud))
print('Generator metadata keys:', sorted(source_cloud['version']['generator_metadata']))
display(pd.DataFrame([{'code': int(k), 'generator_meaning': v}
                     for k, v in source_cloud['version']['generator_metadata']['association_labels'].items()]))
print('Kinect camera calibration top-level keys:',
      sorted(selection['kinect']['camera_calibration']['document']))
print('Framework source hashes are saved in FRAMEWORK_SOURCE_SNAPSHOT for this guide version.')

## 14. Acceptance checks for your adapter

Before training, keep automated checks for these concrete invariants:

1. Payload hashes/schema match; descriptor and adapter-policy versions are persisted.
2. Row 0 maps to its true native sample, `pose_row=-1` never selects the final pose,
   ragged lengths match counts, and empty versus missing remains distinguishable.
3. Accepted radar↔pose matches share a block and satisfy the recorded tolerance;
   model residual is never reported as measured synchronization accuracy.
4. Coordinate round trips pass; XYZ and targets use the same sensor frame/metres;
   Doppler/SNR are unchanged; association codes are not anonymous model features.
5. Every framework sequence starts at frame 0, increments by one, has increasing
   physical times, and cannot bridge original blocks or removed acquisition gaps.
6. Targets are finite even under false masks, the root is valid, multi-person/missing
   cases have explicit outcomes, and every joint mapping is named and versioned.
7. No subjects/related windows cross splits, no held-out labels fit bounds, and target
   selection cannot leak ground truth into a deployment-only crop/anchor policy.
8. A deterministic small batch passes preprocessing, collation, model output-shape and
   masked-loss checks before a full experiment. Report window/joint coverage and reasons
   for rejection, not only the loss of selected examples.

Still requiring a study/experiment decision: whether the initial synchronization is
accurate enough for the intended task; confidence/person/temporal policy; approved
joint topology or cross-dataset correspondence; calibration applicability across
sessions; a two-subject development/evaluation protocol; and a capture-mode camera
model if 3D-to-image projection is added. The exported data and this notebook preserve
the evidence needed to make those decisions rather than silently choosing them.